# `ErrorAnalysis` module

The `bhm.ErrorAnalysis` module provides functions for assessing the error of an optimized model.

Note that these functions all asume the input model has been optimized, if given a non-optimized model,
the results will likely be confusing and in some way incorrect.

First lets load some data to use for demonstration.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import tables as tb
import smfbursts as smf

import H2MMbursts as bhm


raw = smf.photonHDF5.load('data/HP3_TE300_SPC630.hdf5')
data = smf.photonHDF5.regularize_dets(raw)

# load saved data, if it exists
import os
statepathfile = 'data/statepathparams.hdf5'
if os.path.exists(statepathfile):
    with tb.open_file(statepathfile) as f:
        statepaths_hp3 = [smf.Param.decode_group(g) for g in
                          f.list_nodes(f.root.HP3_TE300_SPC630.statepath_params)]
else:
    dbs = smf.fretfactory.make_bg(data, period=60.0, tail_min=5e-4, 
                                  auto_threshold=True, 
                                  func=smf.bg.exp_mlefit, F_bg=1.7)
    smf.fretfactory.make_burst_search(bg=dbs['bg'], m=10, F=6.0, update=dbs)
    gate_All = smf.make_geq_gate(dbs['NphDex_raw'], 50)
    gate_All = gate_All & smf.make_geq_gate(dbs['NphAA_raw'], 25)
    smf.fretfactory.apply_gate(dbs, gate_All)
    streams=(smf.PhSel('0ex0em'), smf.PhSel('0ex1em'), smf.PhSel('1ex1em'))
    statepaths_hp3 = bhm.StatePath.optimize_models(
        data, dbs['bursts'], streams=streams, max_states=8,
        min_states=1, to_state=4, conv_crit='BICph')

## Loglik Error

The first way to assess the error, and the generally prefered way,
is to compute the point at which the loglikelihood drops by some amount when varying a particular parameter.


### Basic Evaluation

The most common way to evaluate the loglik error is to use the `bhm.error.statepath_ll_error()`.
The basic signature is `bhm.error.statepath_ll_error(data, statepath, adjust='trans')`.

The `data` argument should be the source data, a `smf.PhotonData` or `smf.PhotonDataList` object,
and the `statepath` argument should be a 

This function will return two arrays, specifying the low and high (in that order)
values for the error around the index in the specified model array (default is the trans array).

The low and high error is evaluated as the model where, when its value is decrease/increased 
(for the low and high arrays respectively), the loglikelihood decreases by `0.5` of the starting model.

In [2]:
tel, teh = bhm.error.statepath_ll_error(data, statepaths_hp3[3], adjust='trans')
tel, teh

(array([[9.99966969e-01, 9.45262811e-06, 2.23406897e-06, 1.81448522e-05],
        [7.58536108e-06, 9.99968639e-01, 2.45181693e-06, 1.82147384e-05],
        [9.24842346e-07, 1.26774926e-06, 9.99980207e-01, 1.59361406e-05],
        [4.60982339e-06, 4.07436486e-06, 7.78160635e-06, 9.99982329e-01]]),
 array([[9.99969019e-01, 1.08744109e-05, 3.29271908e-06, 2.00718534e-05],
        [8.94001193e-06, 9.99970586e-01, 3.55171232e-06, 2.00811439e-05],
        [1.45014167e-06, 1.80731084e-06, 9.99981364e-01, 1.70761279e-05],
        [5.11809499e-06, 4.55085423e-06, 8.37142927e-06, 9.99983176e-01]]))

By subtracting the original value of the trans array, we get the amount of change.

In [3]:
trans = statepaths_hp3[3].params['model'].trans
tel - trans, teh - trans

(array([[-1.03567562e-06, -6.99860704e-07, -5.12200767e-07,
         -9.51986510e-07],
        [-6.66361506e-07, -9.83350745e-07, -5.36946227e-07,
         -9.22716003e-07],
        [-2.53679523e-07, -2.63102556e-07, -5.83668187e-07,
         -5.63912267e-07],
        [-2.51017525e-07, -2.35261798e-07, -2.92432620e-07,
         -4.26687130e-07]]),
 array([[1.01432321e-06, 7.21922037e-07, 5.46449352e-07, 9.75014669e-07],
        [6.88289343e-07, 9.64018643e-07, 5.62949169e-07, 9.43689456e-07],
        [2.71619798e-07, 2.76459025e-07, 5.72965083e-07, 5.76075070e-07],
        [2.57254073e-07, 2.41227566e-07, 2.97390295e-07, 4.20136801e-07]]))

Averaging these, you can express an error in a more common looking $\pm$ style:

In [4]:
# compute the average +/- error from low/high trans arrays
terrav = (teh - tel) / 2 # compute average difference around trans

# loop below uses the end argument of print to give an array like appearance
for i in range(trans.shape[0]):
    print('[', end='')
    for j in range(trans.shape[1]):
        # use the :e of f-string to force scientific notation with set widths
        print(f'{trans[i,j]:1.4e} +/- {terrav[i,j]:1.4e}',
              # if statement makes newline only when at end of a row
              end=', ' if j+1 != trans.shape[1] else ']\n')

[9.9997e-01 +/- 1.0250e-06, 1.0152e-05 +/- 7.1089e-07, 2.7463e-06 +/- 5.2933e-07, 1.9097e-05 +/- 9.6350e-07]
[8.2517e-06 +/- 6.7733e-07, 9.9997e-01 +/- 9.7368e-07, 2.9888e-06 +/- 5.4995e-07, 1.9137e-05 +/- 9.3320e-07]
[1.1785e-06 +/- 2.6265e-07, 1.5309e-06 +/- 2.6978e-07, 9.9998e-01 +/- 5.7832e-07, 1.6500e-05 +/- 5.6999e-07]
[4.8608e-06 +/- 2.5414e-07, 4.3096e-06 +/- 2.3824e-07, 8.0740e-06 +/- 2.9491e-07, 9.9998e-01 +/- 4.2341e-07]


Changing the `adjust` argument to `adjust="prior"` or `adjust="obs"`, 
we can evaluate the same for the prior and obs arrays:

In [5]:
pel, peh = bhm.error.statepath_ll_error(data, statepaths_hp3[3], adjust='prior')
oel, oeh = bhm.error.statepath_ll_error(data, statepaths_hp3[3], adjust='obs')
oel, oeh

(array([[0.07583581, 0.06830048, 0.84906987],
        [0.85664811, 0.07525927, 0.06160773],
        [0.15158204, 0.29685881, 0.5462329 ],
        [0.45131276, 0.09227517, 0.45287635]]),
 array([[0.08027792, 0.07190634, 0.85463916],
        [0.86186527, 0.07876522, 0.06588829],
        [0.154695  , 0.30051276, 0.55013122],
        [0.45412852, 0.09381265, 0.45559266]]))

### Basic customization

#### Adjusting the target loglik

The target of having a loglikelihood $0.5$ less than the initial is not set in stone.
This value can be changed with the `targ` keyword argument.
The default is $0.5$, and this is normally the sensible value, representing the error,
however, if you want bounds that represent some ofther threshold, and not the standard error,
changing this value can be useful.

In [6]:
tel, teh = bhm.error.statepath_ll_error(data, statepaths_hp3[3], targ=0.7)
tel, teh

(array([[9.99966777e-01, 9.32777843e-06, 2.14388633e-06, 1.79729552e-05],
        [7.46498362e-06, 9.99968456e-01, 2.35653884e-06, 1.80474684e-05],
        [8.80273479e-07, 1.22106704e-06, 9.99980099e-01, 1.58322090e-05],
        [4.56484704e-06, 4.03191092e-06, 7.72925070e-06, 9.99982250e-01]]),
 array([[9.99969202e-01, 1.10083898e-05, 3.39651506e-06, 2.02545963e-05],
        [9.06916036e-06, 9.99970761e-01, 3.65804397e-06, 2.02580909e-05],
        [1.50179380e-06, 1.85937727e-06, 9.99981468e-01, 1.71823109e-05],
        [5.16545510e-06, 4.59555210e-06, 8.42720154e-06, 9.99983252e-01]]))

#### `flatten` argument

By setting the keyword argument `flatten=False`, 
the return value is now a pair of low/high arrays, 
with the values being the full `h2mm_model` that has the reduced loglikelihood.

In [7]:
tel, teh = bhm.error.statepath_ll_error(data, statepaths_hp3[3], 
                                        adjust='trans', flatten=False)
# tel and teh are now arrays of models
teh[0,1]

nstate: 4, ndet: 3, nphot: 350274, niter: 0, loglik: -321530.44925670367 converged state: 0x2
prior:
0.09180008464206446, 0.14192610716122736, 0.24635708795371888, 0.5199167202429893
trans:
0.9999672824806755, 1.0874410854084184e-05, 2.7462697332324526e-06, 1.909683873726998e-05
8.25172258984347e-06, 0.9999696220598387, 2.9887631522342066e-06, 1.913745441912435e-05
1.1785218691734586e-06, 1.530851813100379e-06, 0.9999807905734902, 1.6500052827459896e-05
4.8608409182432066e-06, 4.309626662595637e-06, 8.074038970015473e-06, 0.9999827554934492
obs:
0.07804743236692008, 0.07009259344021249, 0.8518599741928675
0.8592637589574253, 0.07700156051527302, 0.0637346805273017
0.153134712641781, 0.2986820101455996, 0.5481832772126195
0.45271986441892176, 0.09304328947196071, 0.4542368461091176

We can see the change in the model by subtracting the original parameter matrix,
in the case above, this would be the `trans` matrix

In [8]:
teh[0,1].trans - trans

array([[-7.21922037e-07,  7.21922037e-07,  0.00000000e+00,
         0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00]])

Note that the values at the selected index are adjusted lower/higher,
in the first/second returned array respectively, while the rest of the
values the row of the selected index are adjusted to ensure the array
remains row stochastic (sum to 1), in other words, in the oposite direction.

#### `loc` argument

So far all the examples given, multiple iterations are required,
one for each element of the prior, trans, or obs arrays 
(depending on the `adjust` option).

If you need only a single parameter, you can pass the `loc` argument.
Here simply pass the 2-tuple of the index in the array (an int for the prior array).

> **Note**
>
> When the `loc=<<not None>>`, the `flatten` argument
> must be either `False` or `"auto"` (`"auto"` is the default).
> As the error is no longer computed across the proir, trans, or obs arrays.
> The `"auto"` option does as expected.
> I.e. it returns the models if a single optimization is selected,
> and detects which array is being optimized if the function is computing
> multiple optimizations.

In [9]:
tel, teh = bhm.error.statepath_ll_error(data, statepaths_hp3[3], 
                                        adjust='trans', loc=(0,1))
tel

nstate: 4, ndet: 3, nphot: 350274, niter: 0, loglik: -321530.44891122566 converged state: 0x2
prior:
0.09180008464206446, 0.14192610716122736, 0.24635708795371888, 0.5199167202429893
trans:
0.9999687042523571, 9.452623884932684e-06, 2.746271655269845e-06, 1.9096852102613405e-05
8.25172258984347e-06, 0.9999696220598387, 2.9887631522342066e-06, 1.913745441912435e-05
1.1785218691734586e-06, 1.530851813100379e-06, 0.9999807905734902, 1.6500052827459896e-05
4.8608409182432066e-06, 4.309626662595637e-06, 8.074038970015473e-06, 0.9999827554934492
obs:
0.07804743236692008, 0.07009259344021249, 0.8518599741928675
0.8592637589574253, 0.07700156051527302, 0.0637346805273017
0.153134712641781, 0.2986820101455996, 0.5481832772126195
0.45271986441892176, 0.09304328947196071, 0.4542368461091176

### Adjust functions

The core function for evaluating the loglikelihood error is `bhm.error.evalutate_ll_error`.
`bhm.error.statepath_ll_error` calls this function internally.

The basic method is to evaluate the loglikelihood for models, generated by an `adjust` function,
where a given parameter has been varied, until the loglikelihood of the adjusted model is
as set amount less than the optimal.
Adjust functions take a model and a floating point value that specify by how much to change the model.
Basic default `adjust`  functions are specified as string "prior", "trans" and "obs".

First the loglike of the input model is evaluated, and from this the target loglikelihood is determined.
This is used to generate a penatly function, where the returned value is the square distance $(ll_{adjusted} - ll_{target})^{2}$
of the adjusted model from the targed.
[scipy.optimize.fminbound](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.fminbound.html)
is used to find this model.

Below is a simple example of how to evaluate the error of a model:

In [10]:
# from the statepath, get the necessary model/arrays
model = statepaths_hp3[3].params['model']
indexes = data.get_table(statepaths_hp3[3])['indexpath']
times = data.get_table(statepaths_hp3[3])['timepath']

# evaluate the error
terrl, terrh = bhm.error.evalutate_ll_error(model, indexes, times,
                                            bhm.error.trans_adjust,
                                            loc=(0,1), targ=0.5)
# display the low/high values of loc
print(terrl.trans[0,1], terrh.trans[0,1])

9.452623885172799e-06 1.0874405952875907e-05


The basic signature is `bhm.error.evaluate_ll_error(model, indexes, times, adjust, targ=0.5)`,

**Core arguments**

1. `model` is the `hm.h2mm_model` to find the error of
2. `indexes` is the $\mathrm{H^{2}MM}$ index array ("indexpath" column of the `StatePath` `Param`)
3. `times` is the $\mathrm{H^{2}MM}$ index array ("timepath" column of the `StatePath` `Param`)
4. `adjust` a function with the signature `adjust(model, val, **kwargs)`

**optimization arguments**

It should also be noted that `bhm.error.evaluate_ll_error()` also has the `bound_kwargs`, passed to
[scipy.optimize.fminbound](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.fminbound.html) 
keyword argument, as `scipy.optimize.fminbound` is used to find the model loglik decreased by `targ`.
The function given to `scipy.optimize.fminbound` evaluates $(ll_{opt-targ} - ll_{adj})^{2}$.

This requires calling `hm.h2mm_model.evaluate`, kwargs can be passed to this function through the `eval_kwargs` argument.

### Default `adjust` function kwargs

The `adjust` functions are called in the following way: `adjust(model, val, **kwargs)`

Note that any additional kwargs passed to `bhm.error.evaluate_ll_error()` are forwarded to the `adjust` function.

There are 3 built-in options for `adjust`:
1. `bhm.error.prior_adjust` (can also be speciffied as a string, `"prior"`)
2. `bhm.error.trans_adjust` (can also be speciffied as a string, `"trans"`)
3. `bhm.error.obs_adjust` (can also be speciffied as a string, `"obs"`)

In all three of these cases, they require the `loc` keyword argument.
This specifies the "location" (typically a tuple indexing into the specified array), of the parameter to adjust. 
The location may also be specified as a boolean mask, or sequence of locations if multiple location should be adjusted synchronysly. 
The locations should however all belong to the same row

This location specifies the parameter to be adjusted in the given direction.
However, because rows must be row-stochastic, all other parameters in the given row are adjusted in the opposite direction.
Also, adjustment by `val` is computed such that `val` exists in the interval $(0,1)$, rescaling all values accordingly,
and the lower bound is evaluate on the interval $(0, 0.5)$, and likewise the upper bound evaluated on the interval $(0.5, 1)$

#### `outer` argument

If, instead of adjusting all values in the row by a certain amount, it is also possible to specify the
`outer` keyword argument.
Here, pass a boolean mask , or sequence of locs of all "free" parameters. This must be a superset of locs.

In [11]:
# example as sequence of locs
bhm.error.obs_adjust(model, 0.3, loc=(0,1), outer=((0,0), (0,1))).obs - model.obs

array([[ 6.97925109e-02, -6.97925109e-02,  0.00000000e+00],
       [ 2.22044605e-16,  1.38777878e-17,  1.38777878e-17],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00]])

In [12]:
# example as boolean mask
outer = np.zeros((4,3), dtype=np.bool_)
outer[0,:2] = True
bhm.error.obs_adjust(model, 0.3, loc=(0,1), outer=outer).obs - model.obs

array([[ 6.97925109e-02, -6.97925109e-02,  0.00000000e+00],
       [ 2.22044605e-16,  1.38777878e-17,  1.38777878e-17],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00]])

#### Example: adjusting only values relevant to Proximity Ratio

Combining these we can now show how to evaluate just the error in $^{i}E_{raw}$, leaving $^{2}S_{raw}$.

Note that `bhm.error.statepath_ll_error()` has the `loc` keyword argument built-in, 
if it is not supplied (like the first example), then it iterates over every `loc` in the prior, trans or obs array.

Kwargs handed as kwargs to `bhm.error.evaluate_ll_error()` must be passed through the `adj_kwargs` keyword argument.

In [13]:
elow, ehigh = bhm.error.statepath_ll_error(data, statepaths_hp3[3], 
                                           adjust='obs', loc=(0,1), 
                                           adj_kwargs={'outer':outer})

## BootStrap Error

An alternative to the loglik evaluation, which is generally simpler is the bootstrap method.
Here the data is divided into chuncs, and an independant optimization is conducted on each chunk.
The result is a new model for each chunck, 
and error is defned by the standard error of a given parameter based on the variance of that parameter in each chunck.

This is done useing the `bhm.error.BootStrapError.evaluate()` class method.

Simply pass the data and a `bhm.StatePath` based `smf.Param`, and it will produce a `bhm.BootStrapError` object, which stores the relavant errors.
The number of chuncks is specified with the keyword argument `n`, the default is `n=10`.

In [14]:
bserr = bhm.error.BootStrapError.evaluate(data, statepaths_hp3[3], n=10)

The model converged after 175 iterations
The model converged after 194 iterations
The model converged after 305 iterations
The model converged after 242 iterations
The model converged after 302 iterations
The model converged after 177 iterations
The model converged after 167 iterations
The model converged after 240 iterations
The model converged after 458 iterations
The model converged after 334 iterations


Accessing the error is done using the `err_prior/trans/obs` properties.

In [15]:
bserr.err_prior, bserr.err_trans, bserr.err_obs

(array([0.00605974, 0.00951431, 0.01199174, 0.0141201 ]),
 array([[1.16488400e-06, 9.19096366e-07, 4.87602609e-07, 1.48728632e-06],
        [7.92208061e-07, 2.22717765e-06, 6.49602607e-07, 2.61210234e-06],
        [1.55352076e-07, 3.12707638e-07, 1.00957460e-06, 1.00588556e-06],
        [2.77011008e-07, 4.55167735e-07, 5.73256873e-07, 8.19386273e-07]]),
 array([[0.07483015, 0.00221965, 0.07609387],
        [0.07440478, 0.001425  , 0.07434516],
        [0.00128172, 0.00170453, 0.0025162 ],
        [0.00200295, 0.00064494, 0.00183823]]))

The standard deviations can also be accessed through analagous
`std_prior/trans/obs` properties:

In [16]:
bserr.std_prior, bserr.std_trans, bserr.std_obs

(array([0.01916259, 0.0300869 , 0.03792122, 0.04465169]),
 array([[3.68368664e-06, 2.90643791e-06, 1.54193484e-06, 4.70321231e-06],
        [2.50518185e-06, 7.04295412e-06, 2.05422381e-06, 8.26019288e-06],
        [4.91266399e-07, 9.88868379e-07, 3.19255521e-06, 3.18088942e-06],
        [8.75985721e-07, 1.43936676e-06, 1.81279740e-06, 2.59112691e-06]]),
 array([[0.2366337 , 0.00701915, 0.24062995],
        [0.23528859, 0.00450623, 0.23510002],
        [0.00405314, 0.00539021, 0.00795694],
        [0.00633388, 0.00203949, 0.00581299]]))

There are also methods `bhh.error.BootStrapError.col_std()` and `bhh.error.BootStrapError.col_error()`,
where a `smf.Column` object is handed as the only argument, and the function will internally
use `bhm.StatePathBase.model_value` to evaluate the expected error of the given `smf.Column`.
Note that this requires the `smf.Column` has a method for evaluating the expected value from a
$\mathrm{H^{2}MM}$ model.

In [17]:
E = smf.Column(statepaths_hp3[3].base_param, 'E_raw')

bserr.col_std(E), bserr.col_error(E)

(array([0.11888539, 0.12707773, 0.00568323, 0.00430283]),
 array([0.03759486, 0.04018551, 0.00179719, 0.00136067]))

In [18]:
[m.obs[:,:2].sum(axis=1) for m in bserr.models]

[array([0.15682716, 0.92998613, 0.44326286, 0.54329057]),
 array([0.1609359 , 0.94263664, 0.4593819 , 0.55369637]),
 array([0.14002691, 0.94089244, 0.44103354, 0.54385766]),
 array([0.13795448, 0.93650006, 0.44089546, 0.55204093]),
 array([0.13398802, 0.95682052, 0.46224485, 0.54659166]),
 array([0.94944403, 0.15175414, 0.44552964, 0.54760519]),
 array([0.12936007, 0.94203948, 0.45129491, 0.54620447]),
 array([0.15291751, 0.92727667, 0.46299495, 0.54755473]),
 array([0.16591773, 0.91623019, 0.45210973, 0.53106928]),
 array([0.15674255, 0.91777289, 0.45110279, 0.5441266 ])]

In [19]:
np.array([m.obs[:,1] / m.obs[:,:2].sum(axis=1) for m in bserr.models]).std(axis=0)

array([0.11888539, 0.12707773, 0.00568323, 0.00430283])

Finally, it should be noted that the `bhm.error.BootStrapError` object stores the creating `smf.Param`
in the `bhm.error.BootStrapError.param` attribute, 
and each evaluated model, as a tuple in the `bhm.error.BootStrapError.models` attribute:

In [20]:
bserr.param.params

(('model',
  nstate: 4, ndet: 3, nphot: 0, niter: 0, loglik: -inf converged state: 0x8000
  prior:
  0.09180008464206446, 0.14192610716122736, 0.24635708795371888, 0.5199167202429893
  trans:
  0.9999680044027129, 1.0152488816594678e-05, 2.7462697332324526e-06, 1.909683873726998e-05
  8.25172258984347e-06, 0.9999696220598387, 2.9887631522342066e-06, 1.913745441912435e-05
  1.1785218691734586e-06, 1.530851813100379e-06, 0.9999807905734902, 1.6500052827459896e-05
  4.8608409182432066e-06, 4.309626662595637e-06, 8.074038970015473e-06, 0.9999827554934492
  obs:
  0.07804743236692008, 0.07009259344021249, 0.8518599741928675
  0.8592637589574251, 0.077001560515273, 0.06373468052730169
  0.153134712641781, 0.2986820101455996, 0.5481832772126195
  0.45271986441892176, 0.09304328947196071, 0.4542368461091176),
 ('streams',
  (<class 'smfbursts.ph_sel.PhSel'>
   <class 'smfbursts.ph_sel.PhStream'>
   ex = (True):(0)
   em = (True):(0)
   pol = all
   split = all,
   <class 'smfbursts.ph_sel.PhSe

In [21]:
bserr.models[-1]

nstate: 4, ndet: 3, nphot: 0, niter: 0, loglik: -inf converged state: 0x8000
prior:
0.06539700302136885, 0.18986032618037413, 0.29552762355126816, 0.44921504724698885
trans:
0.9999713911029825, 4.5583875854187075e-06, 3.047105494571505e-07, 2.3745798882744263e-05
6.0240368639411525e-06, 0.9999663171570135, 6.086857778289753e-07, 2.7050120344846607e-05
6.275925950949549e-07, 1.0409580463184479e-06, 0.9999838124724064, 1.4518976952144178e-05
5.300323232997938e-06, 5.411355470336492e-06, 6.316071173965917e-06, 0.9999829722501228
obs:
0.07933386164289946, 0.07740869331475529, 0.8432574450423452
0.8397670203551095, 0.07800587335762922, 0.08222710628726135
0.1570835149927343, 0.2940192781725849, 0.5488972068346809
0.4501867897198932, 0.09393981396389821, 0.45587339631620866

## Discusion: Loglikelihood vs BootStrapError

Advantages of LL error
- No optimizations, can be faster to evaluate
- Decrease in loglikelihood provides clear threshold

Advantages of BootStrapError
- Evaluates all parameters together, so correlations can be more easily observed

In general the LL error is prefered because it is a more well defined evaluation.